In [1]:
import pandas as pd
import numpy as np
import os
#import seaborn
import random
from random import randrange
import datetime

In [2]:
#data_folder
folder_icv = "C:/Users/fajmi/OneDrive - GOUVCI/CAE_INS - Fichiers de Cellule d'Analyses Economiques ( CAE)/ICV"
sample_icv = pd.read_stata(os.path.join(folder_icv,"data_out","Echantillon_Global_ICV.dta"))

In [7]:
#Fonctions

def knuth_shuffle(x):
    "Cette fonction sert au mix des valeurs pour définir les ordres aléatoires d'introduction des zd dans les sous-echantillons"
    for i in range(len(x)-1, 0, -1):
        j = randrange(i + 1)
        x[i], x[j] = x[j], x[i]
    return x

def assign_values_iteratively(df, column_name, list_alea):
    'Cette fonction sert à assigner les valeurs aléatoires aux ZDs tirées'
    # Assigner les valeurs initiales
    for i in range(len(list_alea)+1, len(df)+1):
        list_alea.append(list_alea[i - len(knuth_shuffle(list(range(1, 24))))-1])
    df_temp = pd.DataFrame({column_name : list_alea})
     
    # Calculer et assigner les valeurs suivantes itérativement
    df[column_name] = df_temp[column_name].values

     
    return df

In [ ]:
# Calculer les effectifs par région
tableau_region = sample_icv['region'].value_counts().reset_index()
tableau_region.columns = ['Région', 'Effectif']

# Ajouter les proportions et formatage
tableau_region['Pourcentage'] = (tableau_region['Effectif'] / len(sample_icv) * 100).round(2)
tableau_region['Pourcentage Cumulé'] = tableau_region['Pourcentage'].cumsum()

# Affichage stylisé
from IPython.display import display
display(tableau_region.style
        .format({'Pourcentage': '{:.2f}%', 'Pourcentage Cumulé': '{:.2f}%'})
        .set_caption("Répartition des observations par région"))

In [3]:
import pandas as pd
import numpy as np

def assigner_sous_echantillons(groupe_region):
    # Étape 0 : TRI initial par milieu_residence et pcexp
    groupe_trie = groupe_region.sort_values(['milieu', 'pcexp'])
    
    # Étape 1 : Mélange aléatoire (Knuth/Fisher-Yates)
    groupe_melange = groupe_trie.sample(frac=1, random_state=123).reset_index(drop=True)
    
    # Étape 2 : Calcul des tailles de groupes
    n = len(groupe_melange)
    taille_base, reste = divmod(n, 24)
    
    # Étape 3 : Génération des IDs de groupes équilibrés
    ids_groupes = np.concatenate([
        np.full(taille_base + 1, i + 1) for i in range(reste)
    ] + [
        np.full(taille_base, i + 1) for i in range(reste, 24)
    ])
    
    # Étape 4 : Randomisation finale des IDs
    np.random.seed(456)  # Seed supplémentaire pour la cohérence
    np.random.shuffle(ids_groupes)
    
    # Étape 5 : Assignation au DataFrame
    groupe_melange['sous_echantillon'] = ids_groupes
    
    return groupe_melange

# Application à votre DataFrame "sample_icv"
df_resultat = (
    sample_icv
    .groupby('region', group_keys=False)
    .apply(assigner_sous_echantillons)
    .sample(frac=1, random_state=789)  # Shuffle global final optionnel
)

In [4]:
# Utiliser pandas.crosstab pour créer la table
cross_table = pd.crosstab(
    index=df_resultat['region'],       # Lignes = régions
    columns=df_resultat['sous_echantillon'],  # Colonnes = sous-échantillons
    margins=True,                      # Ajouter une ligne "All" pour les totaux
    margins_name="Total"               # Nommer la ligne de totaux
)

print(cross_table)

sous_echantillon         1    2    3    4    5    6    7    8    9   10  ...  \
region                                                                   ...   
1. AUTONOME D'ABIDJAN   32   32   32   32   32   32   32   32   32   32  ...   
2. HAUT-SASSANDRA       20   20   20   20   20   20   20   20   20   20  ...   
3. PORO                 15   15   15   15   15   15   15   15   15   15  ...   
4. GBEKE                17   17   17   17   17   17   17   17   17   17  ...   
5. INDENIE-DJUABLIN     13   13   13   13   13   13   13   13   13   13  ...   
6. TONKPI               17   17   17   17   17   17   17   17   17   17  ...   
7. YAMOUSSOUKRO         11   11   11   11   11   11   11   11   11   11  ...   
8. GONTOUGO             14   14   14   14   14   14   14   14   14   14  ...   
9. SAN-PEDRO            15   15   15   15   15   15   15   15   15   15  ...   
10. KABADOUGOU          10   10   10   10   10   10   10   10   10   10  ...   
11. N'ZI                10   10   10   1

In [5]:
# 1. Générer la cross-table (sans les totaux pour la vérification)
cross_table = pd.crosstab(
    df_resultat['region'], 
    df_resultat['sous_echantillon']
)

# 2. Fonction pour détecter les tailles inégales par région
def verif_tailles(group_sizes):
    unique_sizes = group_sizes.unique()
    return len(unique_sizes) > 1  # True si tailles différentes

# 3. Appliquer la vérification
tailles_inégales = cross_table.apply(verif_tailles, axis=1)

# 4. Créer un rapport détaillé
rapport = pd.DataFrame({
    'Région': cross_table.index,
    'Inégalités détectées': tailles_inégales,
    'Tailles des groupes': [cross_table.loc[region].value_counts().to_dict() for region in cross_table.index]
})

print("=== Rapport d'inégalités ===")
print(rapport)

# 5. Message explicite
if tailles_inégales.any():
    regions_problemes = tailles_inégales[tailles_inégales].index.tolist()
    print(f"\n⚠️ Alerte : Les régions {regions_problemes} ont des sous-échantillons de tailles inégales.")
    print("Cause probable : Le nombre d'observations dans la région n'est pas divisible par 24.")
    print("Solution : C'est normal ! Les premiers 'n' groupes ont +1 élément (où n = reste de la division par 24).")
else:
    print("\n✅ Toutes les régions ont des sous-échantillons de taille parfaitement égale !")

=== Rapport d'inégalités ===
                                      Région  Inégalités détectées  \
region                                                               
1. AUTONOME D'ABIDJAN  1. AUTONOME D'ABIDJAN                 False   
2. HAUT-SASSANDRA          2. HAUT-SASSANDRA                 False   
3. PORO                              3. PORO                 False   
4. GBEKE                            4. GBEKE                 False   
5. INDENIE-DJUABLIN      5. INDENIE-DJUABLIN                 False   
6. TONKPI                          6. TONKPI                 False   
7. YAMOUSSOUKRO              7. YAMOUSSOUKRO                 False   
8. GONTOUGO                      8. GONTOUGO                 False   
9. SAN-PEDRO                    9. SAN-PEDRO                 False   
10. KABADOUGOU                10. KABADOUGOU                  True   
11. N'ZI                            11. N'ZI                  True   
12. MARAHOUE                    12. MARAHOUE                 

In [6]:
df_resultat.to_excel(os.path.join(folder_icv,"data_out", "df_sous_echantillon_ICV.xlsx"), encoding='utf-8')

c:\ProgramData\anaconda3\lib\site-packages\pandas\util\_decorators.py:211: FutureWarning: the 'encoding' keyword is deprecated and will be removed in a future version. Please take steps to stop the use of 'encoding'
  return func(*args, **kwargs)
